# Part 3 — Structured extraction smoke test

> Reorganized from `Airbab2.ipynb`. The original notebook is unchanged. Generated as one of four focused, independently usable parts.


This notebook loads only the 24-comment test sample. API cells remain explicitly gated by the existing confirmation prompt.


In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
TINY_TEST_FILE = BASE_DIR / "reviews_tiny_test_24.csv"

if not TINY_TEST_FILE.exists():
    raise FileNotFoundError(
        "Run 02_sampling.ipynb first to create "
        f"{TINY_TEST_FILE.name}."
    )

df_tiny_24 = pd.read_csv(TINY_TEST_FILE)
print("Loaded test comments:", len(df_tiny_24))


Loaded test comments: 24


## Import the shared extraction schema and prompt

`extraction_config.py` is shared by the test and production notebooks, so the definitions have one source of truth.


In [2]:
from extraction_config import (
    AspectName,
    CommentExtraction,
    EXTRACTION_PROMPT_V1,
    Finding,
)

print("Shared extraction schema and prompt loaded.")


Shared extraction schema and prompt loaded.


### Confirm that the schema works locally

In [3]:
schema_test = CommentExtraction(
    comment_id="test_001",
    findings=[
        Finding(
            aspect="Privacy",
            object="Property",
            observation="The property provided privacy.",
            aspect_score=4,
            severity_score=None,
            evidence_quote="Love the privacy."
        ),
        Finding(
            aspect="Views",
            object="Property",
            observation=(
                "The property offered peaceful and beautiful views."
            ),
            aspect_score=4,
            severity_score=None,
            evidence_quote="Views were peaceful and beautiful."
        ),
    ],
)

print(schema_test.model_dump())

{'comment_id': 'test_001', 'findings': [{'aspect': 'Privacy', 'object': 'Property', 'observation': 'The property provided privacy.', 'aspect_score': 4, 'severity_score': None, 'evidence_quote': 'Love the privacy.'}, {'aspect': 'Views', 'object': 'Property', 'observation': 'The property offered peaceful and beautiful views.', 'aspect_score': 4, 'severity_score': None, 'evidence_quote': 'Views were peaceful and beautiful.'}]}


### Confirm that Draft Prompt 1 was stored

In [4]:
print("Prompt characters:", len(EXTRACTION_PROMPT_V1))
print(EXTRACTION_PROMPT_V1[:300])

Prompt characters: 15225

You are an analytical assistant extracting structured
guest-experience findings from Airbnb reviews.

Read the complete review before extracting findings.
Preserve context between sentences, including contrasts,
pronouns, causes, consequences, and comparisons with the
listing.

Identify every disti


### Select one real review for the first smoke test


In [6]:
test_row = df_tiny_24.iloc[0]

test_comment_id = str(test_row["id"])
test_comment = str(test_row["comments_clean"])

print("Comment ID:")
print(test_comment_id)

print("\nComplete review:")
print(test_comment)

Comment ID:
1220509930957696974

Complete review:
Amazing place! very safe and clean. walkable distance from everything you'll ever need, including touristic spots. Just amazing overall


### Prepare the input for one-review extraction

In [7]:
import json

test_review_input = json.dumps(
    {
        "comment_id": test_comment_id,
        "comment": test_comment
    },
    ensure_ascii=False,
    indent=2
)

print(test_review_input)

{
  "comment_id": "1220509930957696974",
  "comment": "Amazing place! very safe and clean. walkable distance from everything you'll ever need, including touristic spots. Just amazing overall"
}


### Step 10C: Run one real review through the API

In [ ]:
# ---------------------------------------------------------
# Install the OpenAI Python SDK
# ---------------------------------------------------------

# %pip install --upgrade openai

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 14.6 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.49.0
    Uninstalling openai-2.49.0:
      Successfully uninstalled openai-2.49.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# ---------------------------------------------------------
# Set up the OpenAI client
# ---------------------------------------------------------

import os
from getpass import getpass

from openai import OpenAI


if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

# No automatic retries during the development smoke test.
client = OpenAI(max_retries=0)

print("OpenAI client created successfully.")

OpenAI client created successfully.


# ⚠️⚠️⚠️⚠️⚠️ PAID API CELL
### Running this cell and typing RUN sends one API request. Do not include this cell when using Run All.


In [9]:
# ---------------------------------------------------------
# Run one-review structured extraction smoke test
# ---------------------------------------------------------

import openai


# Reset previous results before this test.
test_extraction = None
test_completion = None


confirmation = input(
    "Type RUN to send one paid API request: "
).strip()

if confirmation != "RUN":
    print("API request cancelled.")

else:
    try:
        test_completion = client.chat.completions.parse(
            model="gpt-5-mini-2025-08-07",
            messages=[
                {
                    "role": "system",
                    "content": EXTRACTION_PROMPT_V1,
                },
                {
                    "role": "user",
                    "content": test_review_input,
                },
            ],
            response_format=CommentExtraction,
        )

        test_message = test_completion.choices[0].message

        if test_message.parsed is not None:
            test_extraction = test_message.parsed

            print("Structured extraction completed:\n")
            print(
                test_extraction.model_dump_json(
                    indent=2
                )
            )

        else:
            print("No parsed extraction was returned.")
            print("Refusal:", test_message.refusal)

    except openai.AuthenticationError:
        print(
            "Authentication failed. "
            "Check that your API key is correct."
        )

    except openai.RateLimitError:
        print(
            "The request reached a rate or billing limit. "
            "Check your API account usage and billing."
        )

    except openai.APIConnectionError as error:
        print("The API could not be reached.")
        print(error)

    except openai.APIStatusError as error:
        print("The API returned an error.")
        print("Status code:", error.status_code)
        print("Message:", error)

    except Exception as error:
        print("An unexpected error occurred.")
        print(type(error).__name__, error)

Structured extraction completed:

{
  "comment_id": "1220509930957696974",
  "findings": [
    {
      "aspect": "Safety and security",
      "object": "Property",
      "observation": "The property felt very safe.",
      "aspect_score": 4,
      "severity_score": null,
      "evidence_quote": "very safe and clean."
    },
    {
      "aspect": "Cleanliness",
      "object": "Property",
      "observation": "The property was very clean.",
      "aspect_score": 4,
      "severity_score": null,
      "evidence_quote": "very safe and clean."
    },
    {
      "aspect": "Location",
      "object": "Property",
      "observation": "The property is within walking distance of all needed amenities and tourist spots.",
      "aspect_score": 5,
      "severity_score": null,
      "evidence_quote": "walkable distance from everything you'll ever need, including touristic spots."
    },
    {
      "aspect": "Overall stay",
      "object": "Overall property",
      "observation": "Guest described

In [10]:
# ---------------------------------------------------------
# Confirm that the returned ID matches the input ID
# ---------------------------------------------------------

if test_extraction is not None:
    print("Input comment ID: ", test_comment_id)
    print("Output comment ID:", test_extraction.comment_id)

    if test_extraction.comment_id == test_comment_id:
        print("Comment ID check passed.")
    else:
        print("Warning: The comment IDs do not match.")

Input comment ID:  1220509930957696974
Output comment ID: 1220509930957696974
Comment ID check passed.


In [11]:
# ---------------------------------------------------------
# Display token usage for the smoke test
# ---------------------------------------------------------

if test_completion is not None and test_completion.usage:
    print(
        "Input tokens:",
        test_completion.usage.prompt_tokens,
    )
    print(
        "Output tokens:",
        test_completion.usage.completion_tokens,
    )
    print(
        "Total tokens:",
        test_completion.usage.total_tokens,
    )
else:
    print("No token usage is available.")

Input tokens: 2803
Output tokens: 1056
Total tokens: 3859


## Extraction on the 24-comment test
### Prepare the 24-comment extraction run

In [16]:
import json


TINY_RESULTS_FILE = (
    BASE_DIR / "reviews_tiny_test_24_extractions.jsonl"
)

TINY_ERRORS_FILE = (
    BASE_DIR / "reviews_tiny_test_24_errors.jsonl"
)


# Confirm that the required source columns exist.
required_columns = {
    "tiny_test_order",
    "id",
    "comments_clean",
}

missing_columns = (
    required_columns - set(df_tiny_24.columns)
)

if missing_columns:
    raise KeyError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Confirm that this is still the intended 24-comment test.
if len(df_tiny_24) != 24:
    raise ValueError(
        "Expected 24 test comments, but found "
        f"{len(df_tiny_24)}."
    )


# Confirm that every review ID is unique.
comment_ids = df_tiny_24["id"].astype(str)

if comment_ids.duplicated().any():
    raise ValueError(
        "Duplicate comment IDs were found in the test sample."
    )


# Confirm that every cleaned comment contains text.
empty_comments = (
    df_tiny_24["comments_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

if empty_comments.any():
    raise ValueError(
        "One or more comments_clean values are empty."
    )


# Save the successful one-review smoke test as the first
# completed record, but only when no results file exists yet.
if (
    not TINY_RESULTS_FILE.exists()
    and test_extraction is not None
):
    smoke_record = {
        "tiny_test_order": int(
            test_row["tiny_test_order"]
        ),
        "comment_id": test_comment_id,
        "comments_clean": test_comment,
        "extraction": test_extraction.model_dump(),
        "usage": {
            "input_tokens": (
                test_completion.usage.prompt_tokens
                if test_completion
                and test_completion.usage
                else None
            ),
            "output_tokens": (
                test_completion.usage.completion_tokens
                if test_completion
                and test_completion.usage
                else None
            ),
            "total_tokens": (
                test_completion.usage.total_tokens
                if test_completion
                and test_completion.usage
                else None
            ),
        },
    }

    with TINY_RESULTS_FILE.open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(
                smoke_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    print("Saved the smoke-test result as record 1.")


# Read IDs that have already been completed.
completed_ids = set()

if TINY_RESULTS_FILE.exists():
    with TINY_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                saved_record = json.loads(line)
                completed_ids.add(
                    str(saved_record["comment_id"])
                )

            except (json.JSONDecodeError, KeyError) as error:
                raise ValueError(
                    "Invalid saved result on line "
                    f"{line_number}: {error}"
                ) from error


remaining_ids = set(comment_ids) - completed_ids

print("Total test comments:", len(df_tiny_24))
print("Already completed:", len(completed_ids))
print("Remaining API requests:", len(remaining_ids))
print("Results file:", TINY_RESULTS_FILE.name)
print("Errors file:", TINY_ERRORS_FILE.name)

Total test comments: 24
Already completed: 24
Remaining API requests: 0
Results file: reviews_tiny_test_24_extractions.jsonl
Errors file: reviews_tiny_test_24_errors.jsonl


# ⚠️⚠️⚠️⚠️⚠️  PAID API CELL
## Extract the remaining comments in the 24-comment test

In [ ]:
import json
import openai


# Re-read completed IDs immediately before starting.
completed_ids = set()

if TINY_RESULTS_FILE.exists():
    with TINY_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            if line.strip():
                saved_record = json.loads(line)
                completed_ids.add(
                    str(saved_record["comment_id"])
                )


remaining_rows = (
    df_tiny_24[
        ~df_tiny_24["id"]
        .astype(str)
        .isin(completed_ids)
    ]
    .sort_values("tiny_test_order")
)


print("Already completed:", len(completed_ids))
print("Requests to send:", len(remaining_rows))


confirmation = input(
    "Type RUN 23 to send the remaining paid API requests: "
).strip()


if confirmation != "RUN 23":
    print("API extraction cancelled.")

else:
    successful_count = 0
    error_count = 0
    run_input_tokens = 0
    run_output_tokens = 0
    run_total_tokens = 0

    for progress_number, (_, row) in enumerate(
        remaining_rows.iterrows(),
        start=1,
    ):
        comment_id = str(row["id"])
        comment_text = str(row["comments_clean"])

        review_input = json.dumps(
            {
                "comment_id": comment_id,
                "comment": comment_text,
            },
            ensure_ascii=False,
        )

        print(
            f"\nProcessing {progress_number}/"
            f"{len(remaining_rows)} "
            f"| test order {row['tiny_test_order']} "
            f"| comment ID {comment_id}"
        )

        try:
            completion = client.chat.completions.parse(
                model="gpt-5-mini-2025-08-07",
                messages=[
                    {
                        "role": "system",
                        "content": EXTRACTION_PROMPT_V1,
                    },
                    {
                        "role": "user",
                        "content": review_input,
                    },
                ],
                response_format=CommentExtraction,
            )

            message = completion.choices[0].message

            if message.parsed is None:
                error_record = {
                    "tiny_test_order": int(
                        row["tiny_test_order"]
                    ),
                    "comment_id": comment_id,
                    "error_type": "NoParsedOutput",
                    "message": (
                        "No parsed extraction was returned."
                    ),
                    "refusal": message.refusal,
                }

                with TINY_ERRORS_FILE.open(
                    "a",
                    encoding="utf-8",
                ) as file:
                    file.write(
                        json.dumps(
                            error_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                error_count += 1
                print("No parsed extraction returned.")
                continue

            extraction = message.parsed

            # Confirm that the returned ID is correct.
            if extraction.comment_id != comment_id:
                raise ValueError(
                    "Returned comment ID does not match "
                    f"the input ID: {extraction.comment_id}"
                )

            usage = completion.usage

            result_record = {
                "tiny_test_order": int(
                    row["tiny_test_order"]
                ),
                "comment_id": comment_id,
                "comments_clean": comment_text,
                "extraction": extraction.model_dump(),
                "usage": {
                    "input_tokens": (
                        usage.prompt_tokens
                        if usage
                        else None
                    ),
                    "output_tokens": (
                        usage.completion_tokens
                        if usage
                        else None
                    ),
                    "total_tokens": (
                        usage.total_tokens
                        if usage
                        else None
                    ),
                },
            }

            # Save each success immediately.
            with TINY_RESULTS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        result_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            successful_count += 1

            if usage:
                run_input_tokens += usage.prompt_tokens
                run_output_tokens += usage.completion_tokens
                run_total_tokens += usage.total_tokens

            print(
                "Saved successfully | findings:",
                len(extraction.findings),
            )

        except openai.AuthenticationError as error:
            print("Authentication failed:", error)
            print("Stopping the run.")
            break

        except openai.RateLimitError as error:
            print("Rate or billing limit reached:", error)
            print("Stopping the run. Completed results are saved.")
            break

        except openai.APIConnectionError as error:
            print("API connection failed:", error)
            print("Stopping the run. Completed results are saved.")
            break

        except Exception as error:
            error_record = {
                "tiny_test_order": int(
                    row["tiny_test_order"]
                ),
                "comment_id": comment_id,
                "error_type": type(error).__name__,
                "message": str(error),
            }

            with TINY_ERRORS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        error_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            error_count += 1

            print(
                "Error:",
                type(error).__name__,
                error,
            )


    print("\n--------------- RUN SUMMARY ---------------")
    print("Successful requests:", successful_count)
    print("Errors:", error_count)
    print("Input tokens this run:", run_input_tokens)
    print("Output tokens this run:", run_output_tokens)
    print("Total tokens this run:", run_total_tokens)
    print("Results file:", TINY_RESULTS_FILE.name)
    print("Errors file:", TINY_ERRORS_FILE.name)

Already completed: 24
Requests to send: 0


### Validate the completed 24-comment extraction


In [8]:
import json


result_records = []

with TINY_RESULTS_FILE.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            result_records.append(json.loads(line))

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid result on line {line_number}: {error}"
            ) from error


error_records = []

if TINY_ERRORS_FILE.exists():
    with TINY_ERRORS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                error_records.append(json.loads(line))

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid error record on line "
                    f"{line_number}: {error}"
                ) from error


saved_ids = [
    str(record["comment_id"])
    for record in result_records
]

expected_ids = set(
    df_tiny_24["id"].astype(str)
)

missing_ids = expected_ids - set(saved_ids)

duplicate_ids = {
    comment_id
    for comment_id in saved_ids
    if saved_ids.count(comment_id) > 1
}

total_findings = sum(
    len(record["extraction"]["findings"])
    for record in result_records
)


print("Expected comments:", len(df_tiny_24))
print("Saved result records:", len(result_records))
print("Unique saved IDs:", len(set(saved_ids)))
print("Missing IDs:", sorted(missing_ids))
print("Duplicate IDs:", sorted(duplicate_ids))
print("Error records:", len(error_records))
print("Total extracted findings:", total_findings)


assert len(result_records) == 24, \
    "The results file does not contain exactly 24 records."

assert len(set(saved_ids)) == 24, \
    "The results file does not contain 24 unique comment IDs."

assert not missing_ids, \
    "One or more test comments have no saved extraction."

assert not duplicate_ids, \
    "Duplicate extraction records were found."

assert len(error_records) == 0, \
    "The error file contains one or more errors."


print("\n24-comment extraction validation passed.")

NameError: name 'TINY_RESULTS_FILE' is not defined

### Create a readable table for manual inspection

In [18]:
from typing import get_args

import pandas as pd


allowed_aspects = set(get_args(AspectName))
inspection_rows = []


for record in sorted(
    result_records,
    key=lambda item: item["tiny_test_order"],
):
    outer_comment_id = str(record["comment_id"])
    comment_text = str(record["comments_clean"])

    extraction = record["extraction"]
    returned_comment_id = str(extraction["comment_id"])
    findings = extraction["findings"]

    for finding_number, finding in enumerate(
        findings,
        start=1,
    ):
        aspect_score = finding["aspect_score"]
        severity_score = finding["severity_score"]
        evidence_quote = finding["evidence_quote"]

        if aspect_score < 0:
            severity_rule_passed = (
                severity_score is not None
                and 1 <= severity_score <= 5
            )
        else:
            severity_rule_passed = severity_score is None

        inspection_rows.append({
            "tiny_test_order": record["tiny_test_order"],
            "comment_id": outer_comment_id,
            "finding_number": finding_number,
            "comments_clean": comment_text,
            "aspect": finding["aspect"],
            "object": finding["object"],
            "observation": finding["observation"],
            "aspect_score": aspect_score,
            "severity_score": severity_score,
            "evidence_quote": evidence_quote,

            # Automatic validation checks
            "comment_id_match": (
                returned_comment_id == outer_comment_id
            ),
            "aspect_allowed": (
                finding["aspect"] in allowed_aspects
            ),
            "evidence_exact_match": (
                evidence_quote in comment_text
            ),
            "severity_rule_passed": (
                severity_rule_passed
            ),

            # Empty columns for the later human review
            "manual_decision": "",
            "manual_issue": "",
            "manual_notes": "",
        })


tiny_findings_inspection = pd.DataFrame(
    inspection_rows
).sort_values(
    ["tiny_test_order", "finding_number"]
).reset_index(drop=True)


print(
    "Reviews represented:",
    tiny_findings_inspection["comment_id"].nunique()
)
print(
    "Total findings:",
    len(tiny_findings_inspection)
)
print(
    "Invalid aspects:",
    (~tiny_findings_inspection["aspect_allowed"]).sum()
)
print(
    "Evidence mismatches:",
    (~tiny_findings_inspection["evidence_exact_match"]).sum()
)
print(
    "Severity-rule failures:",
    (~tiny_findings_inspection["severity_rule_passed"]).sum()
)
print(
    "Comment-ID mismatches:",
    (~tiny_findings_inspection["comment_id_match"]).sum()
)


display(
    tiny_findings_inspection[
        [
            "tiny_test_order",
            "finding_number",
            "comments_clean",
            "aspect",
            "object",
            "observation",
            "aspect_score",
            "severity_score",
            "evidence_quote",
            "evidence_exact_match",
            "severity_rule_passed",
        ]
    ]
)

KeyError: 'tiny_test_order'

## Reload saved 24-comment results after a kernel restart
### This cell does not send API requests

In [1]:
from pathlib import Path
from typing import get_args
import json
import pandas as pd

from extraction_config import AspectName


TINY_RESULTS_FILE = (
    Path.cwd() / "reviews_tiny_test_24_extractions.jsonl"
)

allowed_aspects = set(get_args(AspectName))
result_records = []


with TINY_RESULTS_FILE.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            result_records.append(json.loads(line))

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number}: {error}"
            ) from error


inspection_rows = []

for record in sorted(
    result_records,
    key=lambda item: item["tiny_test_order"],
):
    comment_id = str(record["comment_id"])
    comment_text = str(record["comments_clean"])
    extraction = record["extraction"]

    for finding_number, finding in enumerate(
        extraction["findings"],
        start=1,
    ):
        aspect_score = finding["aspect_score"]
        severity_score = finding["severity_score"]
        evidence_quote = finding["evidence_quote"]

        if aspect_score < 0:
            severity_rule_passed = (
                severity_score is not None
                and 1 <= severity_score <= 5
            )
        else:
            severity_rule_passed = severity_score is None

        inspection_rows.append({
            "tiny_test_order": record["tiny_test_order"],
            "comment_id": comment_id,
            "finding_number": finding_number,
            "comments_clean": comment_text,
            "aspect": finding["aspect"],
            "object": finding["object"],
            "observation": finding["observation"],
            "aspect_score": aspect_score,
            "severity_score": severity_score,
            "evidence_quote": evidence_quote,
            "aspect_allowed": (
                finding["aspect"] in allowed_aspects
            ),
            "evidence_exact_match": (
                evidence_quote in comment_text
            ),
            "severity_rule_passed": severity_rule_passed,
        })


tiny_findings_inspection = (
    pd.DataFrame(inspection_rows)
    .sort_values(
        ["tiny_test_order", "finding_number"]
    )
    .reset_index(drop=True)
)


print("Saved reviews loaded:", len(result_records))
print("Findings loaded:", len(tiny_findings_inspection))

Saved reviews loaded: 24
Findings loaded: 90


### Display one review and all of its extracted findings

In [25]:
review_order = 24

review_findings = (
    tiny_findings_inspection[
        tiny_findings_inspection["tiny_test_order"]
        == review_order
    ]
    .sort_values("finding_number")
)

if review_findings.empty:
    print(f"No review found for test order {review_order}.")

else:
    print("Test order:", review_order)
    print("Comment ID:", review_findings.iloc[0]["comment_id"])
    print("\nOriginal comment:\n")
    print(review_findings.iloc[0]["comments_clean"])

    print("\nExtracted findings:")

    display(
        review_findings[
            [
                "finding_number",
                "aspect",
                "object",
                "observation",
                "aspect_score",
                "severity_score",
                "evidence_quote",
            ]
        ]
    )

Test order: 24
Comment ID: 849381100147410457

Original comment:

Would stay here again

Extracted findings:


,finding_number,aspect,object,observation,aspect_score,severity_score,evidence_quote
89,1,Overall stay,Overall property,Guest would stay at the property again,3,NaN,Would stay here again


In [ ]:
# Running all 24 again would cost money and could encourage overfitting 
# to the same small sample.

### Targeted 9-comment regression test

This section checks whether the revised prompt corrected the main issues
identified during the original 24-comment manual review.

In [5]:
TARGET_ORDERS = [
    5,
    8,
    9,
    11,
    12,
    15,
    17,
    20,
    22,
]

df_targeted_9 = (
    df_tiny_24[
        df_tiny_24["tiny_test_order"].isin(TARGET_ORDERS)
    ]
    .copy()
    .sort_values("tiny_test_order")
    .reset_index(drop=True)
)


# Confirm that all 9 intended comments were selected.
if len(df_targeted_9) != 9:
    raise ValueError(
        "Expected 9 targeted comments, but found "
        f"{len(df_targeted_9)}."
    )


selected_orders = (
    df_targeted_9["tiny_test_order"]
    .astype(int)
    .tolist()
)

if selected_orders != TARGET_ORDERS:
    raise ValueError(
        "The selected test orders do not match "
        "the intended targeted set."
    )


# Use new files so the original 24-comment results
# are not overwritten or mixed with this test.
TARGET_RESULTS_FILE = (
    BASE_DIR
    / "reviews_targeted_9_extractions_v2.jsonl"
)

TARGET_ERRORS_FILE = (
    BASE_DIR
    / "reviews_targeted_9_errors_v2.jsonl"
)


print("Targeted comments selected:", len(df_targeted_9))
print("Test orders:", selected_orders)
print("Results file:", TARGET_RESULTS_FILE.name)
print("Errors file:", TARGET_ERRORS_FILE.name)


display(
    df_targeted_9[
        [
            "tiny_test_order",
            "id",
            "comments_clean",
        ]
    ]
)

Targeted comments selected: 9
Test orders: [5, 8, 9, 11, 12, 15, 17, 20, 22]
Results file: reviews_targeted_9_extractions_v2.jsonl
Errors file: reviews_targeted_9_errors_v2.jsonl


,tiny_test_order,id,comments_clean
0,5,1146505471118629299,Our family had a wonderful peaceful time at Jo...
1,8,979093824582645364,The area is a beautiful location on a hill the...
2,9,1422669409460426567,Noah is a really amazing host which is why i m...
3,11,1034192160666558503,A lovely home in an absolutely enchanting loca...
4,12,1182181133718870299,Terrible loss of money. I asked them to reimbu...
5,15,1043689142986767209,This place is just as advertised. Very clean w...
6,17,1558185073092581400,The condo was stunning and very comfortable. V...
7,20,1019058899395941204,Love the privacy. Hannah responds to my messag...
8,22,1234180927290075318,Not only is it important to have a great place...


# ⚠️⚠️⚠️⚠️⚠️ PAID API CELL:
### Extract the targeted 9-comment regression test


In [9]:
import json
import openai


# Re-read completed IDs immediately before starting.
completed_ids = set()

if TARGET_RESULTS_FILE.exists():
    with TARGET_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            if line.strip():
                saved_record = json.loads(line)
                completed_ids.add(
                    str(saved_record["comment_id"])
                )


remaining_rows = (
    df_targeted_9[
        ~df_targeted_9["id"]
        .astype(str)
        .isin(completed_ids)
    ]
    .sort_values("tiny_test_order")
)


print("Already completed:", len(completed_ids))
print("Requests to send:", len(remaining_rows))


confirmation = input(
    "Type RUN 9 to send the targeted paid API requests: "
).strip()


if confirmation != "RUN 9":
    print("API extraction cancelled.")

else:
    successful_count = 0
    error_count = 0
    run_input_tokens = 0
    run_output_tokens = 0
    run_total_tokens = 0

    for progress_number, (_, row) in enumerate(
        remaining_rows.iterrows(),
        start=1,
    ):
        comment_id = str(row["id"])
        comment_text = str(row["comments_clean"])

        review_input = json.dumps(
            {
                "comment_id": comment_id,
                "comment": comment_text,
            },
            ensure_ascii=False,
        )

        print(
            f"\nProcessing {progress_number}/"
            f"{len(remaining_rows)} "
            f"| test order {row['tiny_test_order']} "
            f"| comment ID {comment_id}"
        )

        try:
            completion = client.chat.completions.parse(
                model="gpt-5-mini-2025-08-07",
                messages=[
                    {
                        "role": "system",
                        "content": EXTRACTION_PROMPT_V1,
                    },
                    {
                        "role": "user",
                        "content": review_input,
                    },
                ],
                response_format=CommentExtraction,
            )

            message = completion.choices[0].message

            if message.parsed is None:
                error_record = {
                    "tiny_test_order": int(
                        row["tiny_test_order"]
                    ),
                    "comment_id": comment_id,
                    "error_type": "NoParsedOutput",
                    "message": (
                        "No parsed extraction was returned."
                    ),
                    "refusal": message.refusal,
                }

                with TARGET_ERRORS_FILE.open(
                    "a",
                    encoding="utf-8",
                ) as file:
                    file.write(
                        json.dumps(
                            error_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                error_count += 1
                print("No parsed extraction returned.")
                continue

            extraction = message.parsed

            # Confirm that the returned ID is correct.
            if extraction.comment_id != comment_id:
                raise ValueError(
                    "Returned comment ID does not match "
                    f"the input ID: {extraction.comment_id}"
                )

            usage = completion.usage

            result_record = {
                "tiny_test_order": int(
                    row["tiny_test_order"]
                ),
                "comment_id": comment_id,
                "comments_clean": comment_text,
                "extraction": extraction.model_dump(),
                "usage": {
                    "input_tokens": (
                        usage.prompt_tokens
                        if usage
                        else None
                    ),
                    "output_tokens": (
                        usage.completion_tokens
                        if usage
                        else None
                    ),
                    "total_tokens": (
                        usage.total_tokens
                        if usage
                        else None
                    ),
                },
            }

            # Save each success immediately.
            with TARGET_RESULTS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        result_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            successful_count += 1

            if usage:
                run_input_tokens += usage.prompt_tokens
                run_output_tokens += usage.completion_tokens
                run_total_tokens += usage.total_tokens

            print(
                "Saved successfully | findings:",
                len(extraction.findings),
            )

        except openai.AuthenticationError as error:
            print("Authentication failed:", error)
            print("Stopping the run.")
            break

        except openai.RateLimitError as error:
            print("Rate or billing limit reached:", error)
            print("Stopping the run. Completed results are saved.")
            break

        except openai.APIConnectionError as error:
            print("API connection failed:", error)
            print("Stopping the run. Completed results are saved.")
            break

        except Exception as error:
            error_record = {
                "tiny_test_order": int(
                    row["tiny_test_order"]
                ),
                "comment_id": comment_id,
                "error_type": type(error).__name__,
                "message": str(error),
            }

            with TARGET_ERRORS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        error_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            error_count += 1

            print(
                "Error:",
                type(error).__name__,
                error,
            )


    print("\n--------------- RUN SUMMARY ---------------")
    print("Successful requests:", successful_count)
    print("Errors:", error_count)
    print("Input tokens this run:", run_input_tokens)
    print("Output tokens this run:", run_output_tokens)
    print("Total tokens this run:", run_total_tokens)
    print("Results file:", TARGET_RESULTS_FILE.name)
    print("Errors file:", TARGET_ERRORS_FILE.name)

Already completed: 0
Requests to send: 9

Processing 1/9 | test order 5 | comment ID 1146505471118629299
Saved successfully | findings: 3

Processing 2/9 | test order 8 | comment ID 979093824582645364
Saved successfully | findings: 4

Processing 3/9 | test order 9 | comment ID 1422669409460426567
Saved successfully | findings: 12

Processing 4/9 | test order 11 | comment ID 1034192160666558503
Saved successfully | findings: 3

Processing 5/9 | test order 12 | comment ID 1182181133718870299
Saved successfully | findings: 5

Processing 6/9 | test order 15 | comment ID 1043689142986767209
Saved successfully | findings: 7

Processing 7/9 | test order 17 | comment ID 1558185073092581400
Saved successfully | findings: 7

Processing 8/9 | test order 20 | comment ID 1019058899395941204
Saved successfully | findings: 2

Processing 9/9 | test order 22 | comment ID 1234180927290075318
Saved successfully | findings: 2

--------------- RUN SUMMARY ---------------
Successful requests: 9
Errors: 0
In

In [ ]:
# 1. Group-size language incorrectly interpreted as a noisy party
# 2. Omitted pre-booking restrictions classified as Communication
# instead of Accuracy of listing

## 120-comment development sample
### Load the fresh 120-comment development samplefrom pathlib import Path


In [7]:
import pandas as pd


# ---------------------------------------------------------
# Development sample and output paths
# ---------------------------------------------------------

BASE_DIR = Path.cwd()

DEVELOPMENT_FILE = (
    BASE_DIR / "reviews_development_120.csv"
)

DEVELOPMENT_RESULTS_FILE = (
    BASE_DIR / "reviews_development_120_extractions.jsonl"
)

DEVELOPMENT_ERRORS_FILE = (
    BASE_DIR / "reviews_development_120_errors.jsonl"
)


# ---------------------------------------------------------
# Load the development sample
# ---------------------------------------------------------

if not DEVELOPMENT_FILE.exists():
    raise FileNotFoundError(
        f"Missing file: {DEVELOPMENT_FILE.name}"
    )


df_development_120 = pd.read_csv(
    DEVELOPMENT_FILE
)


# ---------------------------------------------------------
# Validate the loaded sample
# ---------------------------------------------------------

required_columns = {
    "development_order",
    "id",
    "year",
    "listing_activity",
    "comments_clean",
}

missing_columns = (
    required_columns
    - set(df_development_120.columns)
)

if missing_columns:
    raise KeyError(
        "Missing development columns: "
        f"{sorted(missing_columns)}"
    )


assert len(df_development_120) == 120, (
    "The development file does not contain "
    "exactly 120 comments."
)

assert (
    df_development_120["id"]
    .astype(str)
    .nunique()
    == 120
), "The development sample contains duplicate IDs."


df_development_120 = (
    df_development_120
    .sort_values("development_order")
    .reset_index(drop=True)
)


print("Development comments loaded:", len(df_development_120))
print(
    "Unique comment IDs:",
    df_development_120["id"].astype(str).nunique(),
)
print("Results file:", DEVELOPMENT_RESULTS_FILE.name)
print("Errors file:", DEVELOPMENT_ERRORS_FILE.name)

display(
    df_development_120[
        [
            "development_order",
            "id",
            "year",
            "listing_activity",
            "comments_clean",
        ]
    ].head(10)
)

Development comments loaded: 120
Unique comment IDs: 120
Results file: reviews_development_120_extractions.jsonl
Errors file: reviews_development_120_errors.jsonl


,development_order,id,year,listing_activity,comments_clean
0,1,1216130983596413464,2024,Medium,Great hosts and a beautiful spot! David and Ba...
1,2,1052318163965253380,2023,Very high,Really nicely thought out and well-kept space.
2,3,1308129742289085405,2024,Very high,"Clean, organized, easy to get in and out. I'd ..."
3,4,810273047396758029,2023,Medium,thanks for the stay really enjoyed it will boo...
4,5,992119873081175192,2023,Medium,"such a comfortable place to stay, they thought..."
5,6,907383266324818034,2023,Medium,We had a great time.
6,7,1555304315561210533,2025,Low,She was nice
7,8,1134998471789414500,2024,Medium,This place was spacious and clean. Location wa...
8,9,1095877575274830936,2024,Very high,"Great location, great place, great host - woul..."
9,10,966782241452641542,2023,Medium,This was booked for a co-worker traveling for ...


# ⚠️⚠️⚠️⚠️⚠️PAID API CELL:
### Extract the 120-comment development sample

In [8]:
import json

import openai


# ---------------------------------------------------------
# Confirm required objects are available
# ---------------------------------------------------------

if "client" not in globals():
    raise NameError(
        "OpenAI client is not defined. "
        "Run the OpenAI client setup cell first."
    )

if "EXTRACTION_PROMPT_V1" not in globals():
    raise NameError(
        "EXTRACTION_PROMPT_V1 is not defined. "
        "Run the extraction_config import cell first."
    )

if "CommentExtraction" not in globals():
    raise NameError(
        "CommentExtraction is not defined. "
        "Run the extraction_config import cell first."
    )


# ---------------------------------------------------------
# Read IDs that were already completed
# ---------------------------------------------------------

completed_ids = set()

if DEVELOPMENT_RESULTS_FILE.exists():
    with DEVELOPMENT_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            if line.strip():
                saved_record = json.loads(line)

                completed_ids.add(
                    str(saved_record["comment_id"])
                )


# Only send requests for unfinished comments.
remaining_rows = (
    df_development_120[
        ~df_development_120["id"]
        .astype(str)
        .isin(completed_ids)
    ]
    .sort_values("development_order")
)


print("Already completed:", len(completed_ids))
print("Requests to send:", len(remaining_rows))


# ---------------------------------------------------------
# Explicit confirmation before paid requests
# ---------------------------------------------------------

confirmation = input(
    "Type RUN 120 to send the paid API requests: "
).strip()


if confirmation != "RUN 120":
    print("API extraction cancelled.")

else:
    successful_count = 0
    error_count = 0

    run_input_tokens = 0
    run_output_tokens = 0
    run_total_tokens = 0


    # -----------------------------------------------------
    # Process each comment separately
    # -----------------------------------------------------

    for progress_number, (_, row) in enumerate(
        remaining_rows.iterrows(),
        start=1,
    ):
        comment_id = str(row["id"])
        comment_text = str(row["comments_clean"])

        review_input = json.dumps(
            {
                "comment_id": comment_id,
                "comment": comment_text,
            },
            ensure_ascii=False,
        )


        print(
            f"\nProcessing {progress_number}/"
            f"{len(remaining_rows)} "
            f"| development order "
            f"{row['development_order']} "
            f"| comment ID {comment_id}"
        )


        try:
            completion = client.chat.completions.parse(
                model="gpt-5-mini-2025-08-07",
                messages=[
                    {
                        "role": "system",
                        "content": EXTRACTION_PROMPT_V1,
                    },
                    {
                        "role": "user",
                        "content": review_input,
                    },
                ],
                response_format=CommentExtraction,
            )


            message = completion.choices[0].message


            # ---------------------------------------------
            # Handle a response without parsed output
            # ---------------------------------------------

            if message.parsed is None:
                error_record = {
                    "development_order": int(
                        row["development_order"]
                    ),
                    "comment_id": comment_id,
                    "error_type": "NoParsedOutput",
                    "message": (
                        "No parsed extraction was returned."
                    ),
                    "refusal": message.refusal,
                }


                with DEVELOPMENT_ERRORS_FILE.open(
                    "a",
                    encoding="utf-8",
                ) as file:
                    file.write(
                        json.dumps(
                            error_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )


                error_count += 1

                print(
                    "No parsed extraction returned."
                )

                continue


            extraction = message.parsed


            # ---------------------------------------------
            # Confirm the model returned the correct ID
            # ---------------------------------------------

            if extraction.comment_id != comment_id:
                raise ValueError(
                    "Returned comment ID does not match "
                    f"the input ID: "
                    f"{extraction.comment_id}"
                )


            usage = completion.usage


            # ---------------------------------------------
            # Prepare the saved result
            # ---------------------------------------------

            result_record = {
                "development_order": int(
                    row["development_order"]
                ),
                "comment_id": comment_id,
                "year": int(row["year"]),
                "listing_activity": str(
                    row["listing_activity"]
                ),
                "comments_clean": comment_text,
                "extraction": extraction.model_dump(),
                "usage": {
                    "input_tokens": (
                        usage.prompt_tokens
                        if usage
                        else None
                    ),
                    "output_tokens": (
                        usage.completion_tokens
                        if usage
                        else None
                    ),
                    "total_tokens": (
                        usage.total_tokens
                        if usage
                        else None
                    ),
                },
            }


            # Save every successful result immediately.
            with DEVELOPMENT_RESULTS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        result_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )


            successful_count += 1


            if usage:
                run_input_tokens += usage.prompt_tokens
                run_output_tokens += (
                    usage.completion_tokens
                )
                run_total_tokens += usage.total_tokens


            print(
                "Saved successfully | findings:",
                len(extraction.findings),
            )


        # -------------------------------------------------
        # Stop for errors that may affect every request
        # -------------------------------------------------

        except openai.AuthenticationError as error:
            print("Authentication failed:", error)
            print("Stopping the run.")
            break


        except openai.RateLimitError as error:
            print("Rate or billing limit reached:", error)
            print(
                "Stopping the run. "
                "Completed results are already saved."
            )
            break


        except openai.APIConnectionError as error:
            print("API connection failed:", error)
            print(
                "Stopping the run. "
                "Completed results are already saved."
            )
            break


        # -------------------------------------------------
        # Save an individual-comment error and continue
        # -------------------------------------------------

        except Exception as error:
            error_record = {
                "development_order": int(
                    row["development_order"]
                ),
                "comment_id": comment_id,
                "error_type": type(error).__name__,
                "message": str(error),
            }


            with DEVELOPMENT_ERRORS_FILE.open(
                "a",
                encoding="utf-8",
            ) as file:
                file.write(
                    json.dumps(
                        error_record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )


            error_count += 1


            print(
                "Error:",
                type(error).__name__,
                error,
            )


    # -----------------------------------------------------
    # Run summary
    # -----------------------------------------------------

    print("\n--------------- RUN SUMMARY ---------------")
    print("Successful requests:", successful_count)
    print("Errors:", error_count)
    print("Input tokens this run:", run_input_tokens)
    print("Output tokens this run:", run_output_tokens)
    print("Total tokens this run:", run_total_tokens)
    print(
        "Results file:",
        DEVELOPMENT_RESULTS_FILE.name,
    )
    print(
        "Errors file:",
        DEVELOPMENT_ERRORS_FILE.name,
    )

Already completed: 0
Requests to send: 120

Processing 1/120 | development order 1 | comment ID 1216130983596413464
Saved successfully | findings: 8

Processing 2/120 | development order 2 | comment ID 1052318163965253380
Saved successfully | findings: 2

Processing 3/120 | development order 3 | comment ID 1308129742289085405
Saved successfully | findings: 3

Processing 4/120 | development order 4 | comment ID 810273047396758029
Saved successfully | findings: 3

Processing 5/120 | development order 5 | comment ID 992119873081175192
Saved successfully | findings: 4

Processing 6/120 | development order 6 | comment ID 907383266324818034
Saved successfully | findings: 1

Processing 7/120 | development order 7 | comment ID 1555304315561210533
Saved successfully | findings: 1

Processing 8/120 | development order 8 | comment ID 1134998471789414500
Saved successfully | findings: 14

Processing 9/120 | development order 9 | comment ID 1095877575274830936
Saved successfully | findings: 3

Proc

# Rerun the SAME 120 development comments with:
### GPT-5 mini + minimal reasoning + 10 parallel requests


In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from getpass import getpass
import importlib
import json
import os
import time

import pandas as pd
import openai
from openai import OpenAI


# ---------------------------------------------------------
# 1. Reload the latest extraction prompt and schema
# ---------------------------------------------------------

import extraction_config

importlib.reload(extraction_config)

CommentExtraction = extraction_config.CommentExtraction
EXTRACTION_PROMPT_V1 = extraction_config.EXTRACTION_PROMPT_V1

print("Latest extraction_config.py reloaded.")
print("Prompt characters:", len(EXTRACTION_PROMPT_V1))


# ---------------------------------------------------------
# 2. File names and test settings
# ---------------------------------------------------------

BASE_DIR = Path.cwd()

# Your previous 120-comment results.
# This file is used only to recover the exact same comments.
OLD_RESULTS_FILE = (
    BASE_DIR / "reviews_development_120_extractions.jsonl"
)

# New files for the minimal-reasoning test.
NEW_RESULTS_FILE = (
    BASE_DIR
    / "reviews_development_120_minimal_extractions.jsonl"
)

NEW_ERRORS_FILE = (
    BASE_DIR
    / "reviews_development_120_minimal_errors.jsonl"
)

COMPARISON_FILE = (
    BASE_DIR
    / "reviews_development_120_minimal_comparison.csv"
)

MODEL = "gpt-5-mini-2025-08-07"
REASONING_EFFORT = "minimal"

# Ten reviews are processed at the same time,
# but each request still contains only one review.
MAX_WORKERS = 10
MAX_RETRIES = 3

# GPT-5 mini standard prices per 1 million tokens.
INPUT_PRICE = 0.25
CACHED_INPUT_PRICE = 0.025
OUTPUT_PRICE = 2.00


if not OLD_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {OLD_RESULTS_FILE.name}. "
        "Make sure this notebook and the old 120-comment "
        "results are in the same folder."
    )


# ---------------------------------------------------------
# 3. Create a separate API client for this test
# ---------------------------------------------------------

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

test_client = OpenAI(
    max_retries=0,
    timeout=180.0,
)

print("Test API client created.")


# ---------------------------------------------------------
# 4. Load the exact same 120 comments
# ---------------------------------------------------------

old_records = []

with OLD_RESULTS_FILE.open(
    "r",
    encoding="utf-8",
) as file:

    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            old_records.append(
                json.loads(line)
            )

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number} of "
                f"{OLD_RESULTS_FILE.name}: {error}"
            ) from error


source_rows = []

for record in old_records:
    source_rows.append({
        "development_order": int(
            record["development_order"]
        ),
        "comment_id": str(
            record["comment_id"]
        ),
        "year": record.get("year"),
        "listing_activity": record.get(
            "listing_activity"
        ),
        "comments_clean": str(
            record["comments_clean"]
        ),
        "old_extraction": record["extraction"],
    })


df_same_120 = (
    pd.DataFrame(source_rows)
    .sort_values("development_order")
    .reset_index(drop=True)
)


if len(df_same_120) != 120:
    raise ValueError(
        f"Expected 120 comments, "
        f"but found {len(df_same_120)}."
    )


if df_same_120["comment_id"].duplicated().any():
    raise ValueError(
        "Duplicate comment IDs were found."
    )


print("\nSame development comments loaded:", len(df_same_120))
print("Model:", MODEL)
print("Reasoning effort:", REASONING_EFFORT)
print("Parallel workers:", MAX_WORKERS)
print("New results file:", NEW_RESULTS_FILE.name)


# ---------------------------------------------------------
# 5. Helper functions
# ---------------------------------------------------------

def nested_value(
    obj,
    *names,
    default=0,
):
    """
    Safely retrieve nested token-usage values.
    """

    current = obj

    for name in names:
        if current is None:
            return default

        current = getattr(
            current,
            name,
            None,
        )

    if current is None:
        return default

    return current


def calculate_cost(usage):
    """
    Calculate the estimated cost of one request.
    """

    input_tokens = usage["input_tokens"]
    cached_tokens = usage["cached_input_tokens"]
    output_tokens = usage["output_tokens"]

    uncached_tokens = max(
        input_tokens - cached_tokens,
        0,
    )

    cost = (
        uncached_tokens
        * INPUT_PRICE
        / 1_000_000
        +
        cached_tokens
        * CACHED_INPUT_PRICE
        / 1_000_000
        +
        output_tokens
        * OUTPUT_PRICE
        / 1_000_000
    )

    return cost


def extract_one(row):
    """
    Extract one Airbnb review.

    The function retries a failed request up to
    MAX_RETRIES times.
    """

    comment_id = str(
        row["comment_id"]
    )

    comment_text = str(
        row["comments_clean"]
    )

    review_input = json.dumps(
        {
            "comment_id": comment_id,
            "comment": comment_text,
        },
        ensure_ascii=False,
    )


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        started = time.perf_counter()

        try:
            completion = (
                test_client.chat.completions.parse(
                    model=MODEL,

                    reasoning_effort=(
                        REASONING_EFFORT
                    ),

                    messages=[
                        {
                            "role": "system",
                            "content": (
                                EXTRACTION_PROMPT_V1
                            ),
                        },
                        {
                            "role": "user",
                            "content": review_input,
                        },
                    ],

                    response_format=(
                        CommentExtraction
                    ),
                )
            )

            elapsed_seconds = (
                time.perf_counter()
                - started
            )

            message = (
                completion
                .choices[0]
                .message
            )


            if message.parsed is None:
                raise ValueError(
                    "No parsed extraction was returned. "
                    f"Refusal: {message.refusal}"
                )


            extraction = message.parsed


            if str(extraction.comment_id) != comment_id:
                raise ValueError(
                    "Returned comment_id does not "
                    "match the input comment_id."
                )


            usage_object = completion.usage


            usage = {
                "input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens",
                ),

                "cached_input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens_details",
                    "cached_tokens",
                ),

                "output_tokens": nested_value(
                    usage_object,
                    "completion_tokens",
                ),

                "reasoning_tokens": nested_value(
                    usage_object,
                    "completion_tokens_details",
                    "reasoning_tokens",
                ),

                "total_tokens": nested_value(
                    usage_object,
                    "total_tokens",
                ),
            }


            usage["estimated_cost_usd"] = (
                calculate_cost(usage)
            )


            return {
                "status": "success",

                "development_order": int(
                    row["development_order"]
                ),

                "comment_id": comment_id,

                "year": row.get("year"),

                "listing_activity": row.get(
                    "listing_activity"
                ),

                "comments_clean": comment_text,

                "model": MODEL,

                "reasoning_effort": (
                    REASONING_EFFORT
                ),

                "elapsed_seconds": (
                    elapsed_seconds
                ),

                "attempt": attempt,

                "extraction": (
                    extraction.model_dump()
                ),

                "usage": usage,
            }


        except openai.AuthenticationError:
            # An invalid API key will not improve
            # through retries.
            raise


        except Exception as error:

            if attempt < MAX_RETRIES:
                # Wait 2, then 4 seconds.
                time.sleep(
                    2 ** attempt
                )

                continue


            return {
                "status": "error",

                "development_order": int(
                    row["development_order"]
                ),

                "comment_id": comment_id,

                "error_type": (
                    type(error).__name__
                ),

                "message": str(error),

                "attempts": attempt,
            }


# ---------------------------------------------------------
# 6. Resume safely if an earlier test stopped
# ---------------------------------------------------------

completed_ids = set()


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:
            if line.strip():

                saved_record = json.loads(
                    line
                )

                completed_ids.add(
                    str(
                        saved_record[
                            "comment_id"
                        ]
                    )
                )


remaining_rows = df_same_120[
    ~df_same_120["comment_id"].isin(
        completed_ids
    )
].copy()


print(
    "\nAlready completed in minimal test:",
    len(completed_ids),
)

print(
    "Requests to send now:",
    len(remaining_rows),
)


confirmation = input(
    "Type RUN MINIMAL 120 to start the paid test: "
).strip()


# ---------------------------------------------------------
# 7. Run the parallel extraction
# ---------------------------------------------------------

if confirmation != "RUN MINIMAL 120":

    print("API test cancelled.")


elif remaining_rows.empty:

    print(
        "All 120 minimal-reasoning "
        "requests are already complete."
    )


else:
    run_started = time.perf_counter()

    successful_count = 0
    error_count = 0
    run_cost = 0.0

    row_dicts = remaining_rows.to_dict(
        "records"
    )


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                extract_one,
                row,
            ): row["development_order"]

            for row in row_dicts
        }


        with NEW_RESULTS_FILE.open(
            "a",
            encoding="utf-8",
        ) as results_file, NEW_ERRORS_FILE.open(
            "a",
            encoding="utf-8",
        ) as errors_file:


            for finished_number, future in enumerate(
                as_completed(futures),
                start=1,
            ):
                record = future.result()


                if record["status"] == "success":

                    record.pop("status")

                    results_file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    results_file.flush()

                    successful_count += 1

                    run_cost += (
                        record["usage"][
                            "estimated_cost_usd"
                        ]
                    )


                else:
                    record.pop("status")

                    errors_file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    errors_file.flush()

                    error_count += 1


                print(
                    f"Finished {finished_number}/"
                    f"{len(row_dicts)} | "
                    f"success={successful_count} | "
                    f"errors={error_count} | "
                    f"cost=${run_cost:.4f}"
                )


    run_seconds = (
        time.perf_counter()
        - run_started
    )


    print(
        "\n--------------- RUN SUMMARY ---------------"
    )

    print(
        "Successful requests:",
        successful_count,
    )

    print(
        "Errors:",
        error_count,
    )

    print(
        "Wall-clock minutes:",
        round(run_seconds / 60, 2),
    )

    print(
        "Cost for this run: $",
        round(run_cost, 4),
    )


    if successful_count:

        throughput = (
            successful_count
            / run_seconds
        )

        projected_5k_hours = (
            5_000
            / throughput
            / 3600
        )

        projected_50k_hours = (
            50_000
            / throughput
            / 3600
        )

        print(
            "Projected time for 5,000 comments:",
            round(projected_5k_hours, 2),
            "hours",
        )

        print(
            "Projected time for 50,000 comments:",
            round(projected_50k_hours, 2),
            "hours",
        )


# ---------------------------------------------------------
# 8. Reload all new results
# ---------------------------------------------------------

new_records = []


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if line.strip():
                new_records.append(
                    json.loads(line)
                )


# Keep only one result for each comment ID.
new_by_id = {
    str(record["comment_id"]): record
    for record in new_records
}


new_records = sorted(
    new_by_id.values(),
    key=lambda record: (
        record["development_order"]
    ),
)


expected_ids = set(
    df_same_120["comment_id"]
)

returned_ids = set(
    new_by_id
)

missing_ids = (
    expected_ids
    - returned_ids
)


# ---------------------------------------------------------
# 9. Calculate total tokens and price
# ---------------------------------------------------------

all_usage = [
    record["usage"]
    for record in new_records
]


total_cost = sum(
    usage.get(
        "estimated_cost_usd",
        0,
    )
    for usage in all_usage
)


total_input_tokens = sum(
    usage.get(
        "input_tokens",
        0,
    )
    for usage in all_usage
)


total_cached_tokens = sum(
    usage.get(
        "cached_input_tokens",
        0,
    )
    for usage in all_usage
)


total_output_tokens = sum(
    usage.get(
        "output_tokens",
        0,
    )
    for usage in all_usage
)


total_reasoning_tokens = sum(
    usage.get(
        "reasoning_tokens",
        0,
    )
    for usage in all_usage
)


# ---------------------------------------------------------
# 10. Automatic evidence-quote quality check
# ---------------------------------------------------------

evidence_checks = []


for record in new_records:

    validated_extraction = (
        CommentExtraction.model_validate(
            record["extraction"]
        )
    )


    for finding in validated_extraction.findings:

        quote_is_exact = (
            finding.evidence_quote
            in record["comments_clean"]
        )

        evidence_checks.append(
            quote_is_exact
        )


if evidence_checks:

    evidence_pass_rate = (
        sum(evidence_checks)
        / len(evidence_checks)
    )

else:

    evidence_pass_rate = 1.0


print(
    "\n--------------- TOTAL TEST RESULTS ---------------"
)

print(
    "Returned comments:",
    len(new_records),
    "/ 120",
)

print(
    "Missing comments:",
    len(missing_ids),
)

print(
    "Input tokens:",
    total_input_tokens,
)

print(
    "Cached input tokens:",
    total_cached_tokens,
)

print(
    "Output tokens:",
    total_output_tokens,
)

print(
    "Reasoning tokens:",
    total_reasoning_tokens,
)

print(
    "Measured total cost: $",
    round(total_cost, 4),
)

print(
    "Exact evidence-quote pass rate:",
    f"{evidence_pass_rate:.1%}",
)


if new_records:

    projected_5k_cost = (
        total_cost
        / len(new_records)
        * 5_000
    )

    projected_50k_cost = (
        total_cost
        / len(new_records)
        * 50_000
    )

    print(
        "Projected cost for 5,000 comments: $",
        round(projected_5k_cost, 2),
    )

    print(
        "Projected cost for 50,000 comments: $",
        round(projected_50k_cost, 2),
    )


if missing_ids:

    print(
        "Missing comment IDs:",
        sorted(missing_ids),
    )


# ---------------------------------------------------------
# 11. Create old-vs-new reliability comparison
# ---------------------------------------------------------

old_by_id = {
    str(row["comment_id"]): row
    for row in source_rows
}


comparison_rows = []


for comment_id in df_same_120["comment_id"]:

    old_record = old_by_id[
        comment_id
    ]

    new_record = new_by_id.get(
        comment_id
    )


    old_findings = (
        old_record[
            "old_extraction"
        ][
            "findings"
        ]
    )


    if new_record:

        new_findings = (
            new_record[
                "extraction"
            ][
                "findings"
            ]
        )

    else:

        new_findings = []


    comparison_rows.append({
        "development_order": (
            old_record[
                "development_order"
            ]
        ),

        "comment_id": comment_id,

        "comments_clean": (
            old_record[
                "comments_clean"
            ]
        ),

        "old_finding_count": (
            len(old_findings)
        ),

        "minimal_finding_count": (
            len(new_findings)
        ),

        "old_aspects": " | ".join(
            finding["aspect"]
            for finding in old_findings
        ),

        "minimal_aspects": " | ".join(
            finding["aspect"]
            for finding in new_findings
        ),

        "old_extraction_json": json.dumps(
            old_record[
                "old_extraction"
            ],
            ensure_ascii=False,
        ),

        "minimal_extraction_json": (
            json.dumps(
                new_record[
                    "extraction"
                ],
                ensure_ascii=False,
            )
            if new_record
            else ""
        ),
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df.to_csv(
    COMPARISON_FILE,
    index=False,
)


print(
    "\nComparison file saved:",
    COMPARISON_FILE.name,
)


# ---------------------------------------------------------
# 12. Display the orders connected to the issues reviewed
# ---------------------------------------------------------

focus_orders = [
    4,
    5,
    8,
    30,
    31,
    47,
    54,
    55,
    61,
    64,
    77,
    79,
    84,
    85,
    86,
    92,
    96,
    97,
    104,
    111,
    115,
    118,
]


focus_comparison = comparison_df[
    comparison_df[
        "development_order"
    ].isin(
        focus_orders
    )
][[
    "development_order",
    "comments_clean",
    "old_aspects",
    "minimal_aspects",
    "old_finding_count",
    "minimal_finding_count",
]]


focus_comparison

Latest extraction_config.py reloaded.
Prompt characters: 18713
Test API client created.

Same development comments loaded: 120
Model: gpt-5-mini-2025-08-07
Reasoning effort: minimal
Parallel workers: 10
New results file: reviews_development_120_minimal_extractions.jsonl

Already completed in minimal test: 0
Requests to send now: 120
Finished 1/120 | success=1 | errors=0 | cost=$0.0002
Finished 2/120 | success=2 | errors=0 | cost=$0.0005
Finished 3/120 | success=3 | errors=0 | cost=$0.0008
Finished 4/120 | success=4 | errors=0 | cost=$0.0010
Finished 5/120 | success=5 | errors=0 | cost=$0.0024
Finished 6/120 | success=6 | errors=0 | cost=$0.0029
Finished 7/120 | success=7 | errors=0 | cost=$0.0035
Finished 8/120 | success=8 | errors=0 | cost=$0.0039
Finished 9/120 | success=9 | errors=0 | cost=$0.0043
Finished 10/120 | success=10 | errors=0 | cost=$0.0050
Finished 11/120 | success=11 | errors=0 | cost=$0.0053
Finished 12/120 | success=12 | errors=0 | cost=$0.0060
Finished 13/120 | succe

,development_order,comments_clean,old_aspects,minimal_aspects,old_finding_count,minimal_finding_count
3,4,thanks for the stay really enjoyed it will boo...,Other | Overall stay | Overall stay,,3,0
4,5,"such a comfortable place to stay, they thought...",Comfort | Amenities | Property condition | Loc...,Comfort | Amenities | Location,4,3
7,8,This place was spacious and clean. Location wa...,Cleanliness | Comfort | Location | Amenities |...,Space and capacity | Cleanliness | Location | ...,14,10
29,30,Amazing space and great frontdesk,Overall stay | Other,Overall stay | Other,2,2
30,31,"This place was so beautiful,nice and quiet. Th...",Quietness | Location | Cleanliness | Comfort |...,Overall stay | Location | Cleanliness | Comfor...,8,6
46,47,Our family loved our quick stay here. The hous...,Overall stay | Property condition | Amenities ...,Overall stay | Property condition | Communicat...,6,5
53,54,I adored this place! This historic Inn is abso...,Overall stay | Communication | Amenities | Loc...,Overall stay | Communication | Amenities | Loc...,5,5
54,55,If you are looking for a place to stay in Pasa...,Overall stay | Location | Location | Comfort |...,Overall stay | Location | Property condition |...,11,8
60,61,"Wir waren, im Rahmen einer USA Westküsten Rund...",Overall stay | Check-in | Amenities | Accuracy...,Overall stay | Check-in | Amenities | Accuracy...,9,9
63,64,Host was very responsive. Place was nice and q...,Communication | Quietness | Overall stay | Other,Communication | Quietness | Other,4,3


In [ ]:
#The conclusion from this run is:
#Requirement	Minimal reasoning
#Fast	            Yes
#Low price	        Yes
#Reliable quality	No
#so we have to consider antoher method 

# Rerun the SAME 120 development comments with:
### GPT-5 mini + LOW reasoning + 10 parallel requests


In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from getpass import getpass
import importlib
import json
import os
import time

import pandas as pd
import openai
from openai import OpenAI


# ---------------------------------------------------------
# 1. Reload the latest extraction prompt and schema
# ---------------------------------------------------------

import extraction_config

importlib.reload(extraction_config)

CommentExtraction = extraction_config.CommentExtraction
EXTRACTION_PROMPT_V1 = extraction_config.EXTRACTION_PROMPT_V1

print("Latest extraction_config.py reloaded.")
print("Prompt characters:", len(EXTRACTION_PROMPT_V1))


# ---------------------------------------------------------
# 2. File names and test settings
# ---------------------------------------------------------

BASE_DIR = Path.cwd()

# Previous 120-comment results.
# Used only to recover the exact same 120 comments.
OLD_RESULTS_FILE = (
    BASE_DIR / "reviews_development_120_extractions.jsonl"
)

# Separate files for the LOW-reasoning test.
NEW_RESULTS_FILE = (
    BASE_DIR
    / "reviews_development_120_low_extractions.jsonl"
)

NEW_ERRORS_FILE = (
    BASE_DIR
    / "reviews_development_120_low_errors.jsonl"
)

COMPARISON_FILE = (
    BASE_DIR
    / "reviews_development_120_low_comparison.csv"
)

MODEL = "gpt-5-mini-2025-08-07"
REASONING_EFFORT = "low"

# Ten reviews are processed simultaneously,
# but each API request contains only one review.
MAX_WORKERS = 10
MAX_RETRIES = 3

# GPT-5 mini standard prices per 1 million tokens.
INPUT_PRICE = 0.25
CACHED_INPUT_PRICE = 0.025
OUTPUT_PRICE = 2.00


if not OLD_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {OLD_RESULTS_FILE.name}. "
        "Keep the notebook and old 120-comment results "
        "in the same folder."
    )


# ---------------------------------------------------------
# 3. Create an API client
# ---------------------------------------------------------

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

test_client = OpenAI(
    max_retries=0,
    timeout=180.0,
)

print("Test API client created.")


# ---------------------------------------------------------
# 4. Load the exact same 120 comments
# ---------------------------------------------------------

old_records = []

with OLD_RESULTS_FILE.open(
    "r",
    encoding="utf-8",
) as file:

    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            old_records.append(
                json.loads(line)
            )

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number} of "
                f"{OLD_RESULTS_FILE.name}: {error}"
            ) from error


source_rows = []

for record in old_records:
    source_rows.append(
        {
            "development_order": int(
                record["development_order"]
            ),
            "comment_id": str(
                record["comment_id"]
            ),
            "year": record.get("year"),
            "listing_activity": record.get(
                "listing_activity"
            ),
            "comments_clean": str(
                record["comments_clean"]
            ),
            "old_extraction": record["extraction"],
        }
    )


df_same_120 = (
    pd.DataFrame(source_rows)
    .sort_values("development_order")
    .reset_index(drop=True)
)


if len(df_same_120) != 120:
    raise ValueError(
        f"Expected 120 comments, "
        f"but found {len(df_same_120)}."
    )


if df_same_120["comment_id"].duplicated().any():
    raise ValueError(
        "Duplicate comment IDs were found."
    )


print("\nSame development comments loaded:", len(df_same_120))
print("Model:", MODEL)
print("Reasoning effort:", REASONING_EFFORT)
print("Parallel workers:", MAX_WORKERS)
print("New results file:", NEW_RESULTS_FILE.name)


# ---------------------------------------------------------
# 5. Helper functions
# ---------------------------------------------------------

def nested_value(
    obj,
    *names,
    default=0,
):
    """
    Safely retrieve a nested usage value.
    """

    current = obj

    for name in names:
        if current is None:
            return default

        current = getattr(
            current,
            name,
            None,
        )

    if current is None:
        return default

    return current


def calculate_cost(usage):
    """
    Calculate the estimated cost of one request.
    """

    input_tokens = usage["input_tokens"]
    cached_tokens = usage["cached_input_tokens"]
    output_tokens = usage["output_tokens"]

    uncached_tokens = max(
        input_tokens - cached_tokens,
        0,
    )

    return (
        uncached_tokens
        * INPUT_PRICE
        / 1_000_000
        +
        cached_tokens
        * CACHED_INPUT_PRICE
        / 1_000_000
        +
        output_tokens
        * OUTPUT_PRICE
        / 1_000_000
    )


def extract_one(row):
    """
    Extract one Airbnb review using low reasoning.
    Retry up to MAX_RETRIES times if necessary.
    """

    comment_id = str(
        row["comment_id"]
    )

    comment_text = str(
        row["comments_clean"]
    )

    review_input = json.dumps(
        {
            "comment_id": comment_id,
            "comment": comment_text,
        },
        ensure_ascii=False,
    )


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        started = time.perf_counter()

        try:
            completion = (
                test_client.chat.completions.parse(
                    model=MODEL,

                    reasoning_effort=(
                        REASONING_EFFORT
                    ),

                    messages=[
                        {
                            "role": "system",
                            "content": (
                                EXTRACTION_PROMPT_V1
                            ),
                        },
                        {
                            "role": "user",
                            "content": review_input,
                        },
                    ],

                    response_format=(
                        CommentExtraction
                    ),
                )
            )

            elapsed_seconds = (
                time.perf_counter()
                - started
            )

            message = (
                completion
                .choices[0]
                .message
            )


            if message.parsed is None:
                raise ValueError(
                    "No parsed extraction was returned. "
                    f"Refusal: {message.refusal}"
                )


            extraction = message.parsed


            if str(extraction.comment_id) != comment_id:
                raise ValueError(
                    "Returned comment_id does not match "
                    "the input comment_id."
                )


            usage_object = completion.usage


            usage = {
                "input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens",
                ),

                "cached_input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens_details",
                    "cached_tokens",
                ),

                "output_tokens": nested_value(
                    usage_object,
                    "completion_tokens",
                ),

                "reasoning_tokens": nested_value(
                    usage_object,
                    "completion_tokens_details",
                    "reasoning_tokens",
                ),

                "total_tokens": nested_value(
                    usage_object,
                    "total_tokens",
                ),
            }


            usage["estimated_cost_usd"] = (
                calculate_cost(usage)
            )


            return {
                "status": "success",

                "development_order": int(
                    row["development_order"]
                ),

                "comment_id": comment_id,

                "year": row.get("year"),

                "listing_activity": row.get(
                    "listing_activity"
                ),

                "comments_clean": comment_text,

                "model": MODEL,

                "reasoning_effort": (
                    REASONING_EFFORT
                ),

                "elapsed_seconds": (
                    elapsed_seconds
                ),

                "attempt": attempt,

                "extraction": (
                    extraction.model_dump()
                ),

                "usage": usage,
            }


        except openai.AuthenticationError:
            # An invalid API key will not improve
            # through retries.
            raise


        except Exception as error:

            if attempt < MAX_RETRIES:
                # Wait 2 seconds, then 4 seconds.
                time.sleep(
                    2 ** attempt
                )

                continue


            return {
                "status": "error",

                "development_order": int(
                    row["development_order"]
                ),

                "comment_id": comment_id,

                "error_type": (
                    type(error).__name__
                ),

                "message": str(error),

                "attempts": attempt,
            }


# ---------------------------------------------------------
# 6. Resume safely if an earlier low test stopped
# ---------------------------------------------------------

completed_ids = set()


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:
            if not line.strip():
                continue

            saved_record = json.loads(
                line
            )

            completed_ids.add(
                str(
                    saved_record["comment_id"]
                )
            )


remaining_rows = df_same_120[
    ~df_same_120["comment_id"].isin(
        completed_ids
    )
].copy()


print(
    "\nAlready completed in low-reasoning test:",
    len(completed_ids),
)

print(
    "Requests to send now:",
    len(remaining_rows),
)


confirmation = input(
    "Type RUN LOW 120 to start the paid test: "
).strip()


# ---------------------------------------------------------
# 7. Run the parallel extraction
# ---------------------------------------------------------

run_seconds = None


if confirmation != "RUN LOW 120":

    print("API test cancelled.")


elif remaining_rows.empty:

    print(
        "All 120 low-reasoning requests "
        "are already complete."
    )


else:
    run_started = time.perf_counter()

    successful_count = 0
    error_count = 0
    run_cost = 0.0

    row_dicts = remaining_rows.to_dict(
        "records"
    )


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                extract_one,
                row,
            ): row["development_order"]

            for row in row_dicts
        }


        with NEW_RESULTS_FILE.open(
            "a",
            encoding="utf-8",
        ) as results_file, NEW_ERRORS_FILE.open(
            "a",
            encoding="utf-8",
        ) as errors_file:


            for finished_number, future in enumerate(
                as_completed(futures),
                start=1,
            ):
                record = future.result()


                if record["status"] == "success":

                    record.pop("status")

                    results_file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    results_file.flush()

                    successful_count += 1

                    run_cost += (
                        record["usage"][
                            "estimated_cost_usd"
                        ]
                    )


                else:
                    record.pop("status")

                    errors_file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    errors_file.flush()

                    error_count += 1


                print(
                    f"Finished {finished_number}/"
                    f"{len(row_dicts)} | "
                    f"success={successful_count} | "
                    f"errors={error_count} | "
                    f"cost=${run_cost:.4f}"
                )


    run_seconds = (
        time.perf_counter()
        - run_started
    )


    print(
        "\n--------------- RUN SUMMARY ---------------"
    )

    print(
        "Successful requests:",
        successful_count,
    )

    print(
        "Errors:",
        error_count,
    )

    print(
        "Wall-clock minutes:",
        round(run_seconds / 60, 2),
    )

    print(
        "Cost for this run: $",
        round(run_cost, 4),
    )


    if successful_count:

        throughput = (
            successful_count
            / run_seconds
        )

        projected_5k_hours = (
            5_000
            / throughput
            / 3600
        )

        projected_50k_hours = (
            50_000
            / throughput
            / 3600
        )

        print(
            "Projected time for 5,000 comments:",
            round(projected_5k_hours, 2),
            "hours",
        )

        print(
            "Projected time for 50,000 comments:",
            round(projected_50k_hours, 2),
            "hours",
        )


# ---------------------------------------------------------
# 8. Reload all low-reasoning results
# ---------------------------------------------------------

new_records = []


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if line.strip():
                new_records.append(
                    json.loads(line)
                )


# Keep only one result for each comment ID.
new_by_id = {
    str(record["comment_id"]): record
    for record in new_records
}


new_records = sorted(
    new_by_id.values(),
    key=lambda record: (
        record["development_order"]
    ),
)


expected_ids = set(
    df_same_120["comment_id"]
)

returned_ids = set(
    new_by_id
)

missing_ids = (
    expected_ids
    - returned_ids
)


# ---------------------------------------------------------
# 9. Calculate total tokens and price
# ---------------------------------------------------------

all_usage = [
    record["usage"]
    for record in new_records
]


total_cost = sum(
    usage.get(
        "estimated_cost_usd",
        0,
    )
    for usage in all_usage
)


total_input_tokens = sum(
    usage.get(
        "input_tokens",
        0,
    )
    for usage in all_usage
)


total_cached_tokens = sum(
    usage.get(
        "cached_input_tokens",
        0,
    )
    for usage in all_usage
)


total_output_tokens = sum(
    usage.get(
        "output_tokens",
        0,
    )
    for usage in all_usage
)


total_reasoning_tokens = sum(
    usage.get(
        "reasoning_tokens",
        0,
    )
    for usage in all_usage
)


# ---------------------------------------------------------
# 10. Automatic evidence-quote quality check
# ---------------------------------------------------------

evidence_checks = []


for record in new_records:

    validated_extraction = (
        CommentExtraction.model_validate(
            record["extraction"]
        )
    )


    for finding in validated_extraction.findings:

        quote_is_exact = (
            finding.evidence_quote
            in record["comments_clean"]
        )

        evidence_checks.append(
            quote_is_exact
        )


if evidence_checks:

    evidence_pass_rate = (
        sum(evidence_checks)
        / len(evidence_checks)
    )

else:

    evidence_pass_rate = 1.0


print(
    "\n--------------- TOTAL LOW TEST RESULTS ---------------"
)

print(
    "Returned comments:",
    len(new_records),
    "/ 120",
)

print(
    "Missing comments:",
    len(missing_ids),
)

print(
    "Input tokens:",
    total_input_tokens,
)

print(
    "Cached input tokens:",
    total_cached_tokens,
)

print(
    "Output tokens:",
    total_output_tokens,
)

print(
    "Reasoning tokens:",
    total_reasoning_tokens,
)

print(
    "Measured total cost: $",
    round(total_cost, 4),
)

print(
    "Exact evidence-quote pass rate:",
    f"{evidence_pass_rate:.1%}",
)


if new_records:

    projected_5k_cost = (
        total_cost
        / len(new_records)
        * 5_000
    )

    projected_50k_cost = (
        total_cost
        / len(new_records)
        * 50_000
    )

    print(
        "Projected cost for 5,000 comments: $",
        round(projected_5k_cost, 2),
    )

    print(
        "Projected cost for 50,000 comments: $",
        round(projected_50k_cost, 2),
    )


if missing_ids:

    print(
        "Missing comment IDs:",
        sorted(missing_ids),
    )


# ---------------------------------------------------------
# 11. Create old-versus-low reliability comparison
# ---------------------------------------------------------

old_by_id = {
    str(row["comment_id"]): row
    for row in source_rows
}


comparison_rows = []


for comment_id in df_same_120["comment_id"]:

    old_record = old_by_id[
        comment_id
    ]

    new_record = new_by_id.get(
        comment_id
    )


    old_findings = (
        old_record[
            "old_extraction"
        ][
            "findings"
        ]
    )


    if new_record:

        low_findings = (
            new_record[
                "extraction"
            ][
                "findings"
            ]
        )

    else:

        low_findings = []


    comparison_rows.append(
        {
            "development_order": (
                old_record[
                    "development_order"
                ]
            ),

            "comment_id": comment_id,

            "comments_clean": (
                old_record[
                    "comments_clean"
                ]
            ),

            "old_finding_count": (
                len(old_findings)
            ),

            "low_finding_count": (
                len(low_findings)
            ),

            "old_aspects": " | ".join(
                finding["aspect"]
                for finding in old_findings
            ),

            "low_aspects": " | ".join(
                finding["aspect"]
                for finding in low_findings
            ),

            "old_extraction_json": json.dumps(
                old_record[
                    "old_extraction"
                ],
                ensure_ascii=False,
            ),

            "low_extraction_json": (
                json.dumps(
                    new_record[
                        "extraction"
                    ],
                    ensure_ascii=False,
                )
                if new_record
                else ""
            ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df.to_csv(
    COMPARISON_FILE,
    index=False,
)


print(
    "\nComparison file saved:",
    COMPARISON_FILE.name,
)


# ---------------------------------------------------------
# 12. Display orders connected to reviewed issues
# ---------------------------------------------------------

focus_orders = [
    4,
    5,
    8,
    30,
    31,
    47,
    54,
    55,
    61,
    64,
    77,
    79,
    84,
    85,
    86,
    92,
    96,
    97,
    104,
    111,
    115,
    118,
]


focus_comparison = comparison_df[
    comparison_df[
        "development_order"
    ].isin(
        focus_orders
    )
][[
    "development_order",
    "comments_clean",
    "old_aspects",
    "low_aspects",
    "old_finding_count",
    "low_finding_count",
]]


focus_comparison

Latest extraction_config.py reloaded.
Prompt characters: 18713
Test API client created.

Same development comments loaded: 120
Model: gpt-5-mini-2025-08-07
Reasoning effort: low
Parallel workers: 10
New results file: reviews_development_120_low_extractions.jsonl

Already completed in low-reasoning test: 0
Requests to send now: 120
Finished 1/120 | success=1 | errors=0 | cost=$0.0004
Finished 2/120 | success=2 | errors=0 | cost=$0.0018
Finished 3/120 | success=3 | errors=0 | cost=$0.0024
Finished 4/120 | success=4 | errors=0 | cost=$0.0036
Finished 5/120 | success=5 | errors=0 | cost=$0.0050
Finished 6/120 | success=6 | errors=0 | cost=$0.0064
Finished 7/120 | success=7 | errors=0 | cost=$0.0078
Finished 8/120 | success=8 | errors=0 | cost=$0.0093
Finished 9/120 | success=9 | errors=0 | cost=$0.0109
Finished 10/120 | success=10 | errors=0 | cost=$0.0120
Finished 11/120 | success=11 | errors=0 | cost=$0.0125
Finished 12/120 | success=12 | errors=0 | cost=$0.0131
Finished 13/120 | success

,development_order,comments_clean,old_aspects,low_aspects,old_finding_count,low_finding_count
3,4,thanks for the stay really enjoyed it will boo...,Other | Overall stay | Overall stay,Overall stay,3,1
4,5,"such a comfortable place to stay, they thought...",Comfort | Amenities | Property condition | Loc...,Comfort | Amenities | Amenities | Location,4,4
7,8,This place was spacious and clean. Location wa...,Cleanliness | Comfort | Location | Amenities |...,Cleanliness | Space and capacity | Location | ...,14,11
29,30,Amazing space and great frontdesk,Overall stay | Other,Space and capacity | Other,2,2
30,31,"This place was so beautiful,nice and quiet. Th...",Quietness | Location | Cleanliness | Comfort |...,Overall stay | Quietness | Location | Cleanlin...,8,7
46,47,Our family loved our quick stay here. The hous...,Overall stay | Property condition | Amenities ...,Overall stay | Property condition | Communicat...,6,5
53,54,I adored this place! This historic Inn is abso...,Overall stay | Communication | Amenities | Loc...,Overall stay | Other | Amenities | Location,5,4
54,55,If you are looking for a place to stay in Pasa...,Overall stay | Location | Location | Comfort |...,Location | Amenities | Amenities | Comfort | C...,11,10
60,61,"Wir waren, im Rahmen einer USA Westküsten Rund...",Overall stay | Check-in | Amenities | Accuracy...,Overall stay | Check-in | Amenities | Accuracy...,9,9
63,64,Host was very responsive. Place was nice and q...,Communication | Quietness | Overall stay | Other,Communication | Overall stay | Quietness,4,3


In [ ]:
#Requirement	Low reasoning result
#Fast	                Yes
#Reasonable price	    Nearly, but above $50
#Reliable extraction    No(better but still NO)

# SAME 120 comments: GPT-5 mini + MEDIUM reasoning
### 10 comments per request + 10 parallel requests


In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from getpass import getpass
from pydantic import BaseModel, ConfigDict, Field
import importlib
import json
import os
import time

import openai
import pandas as pd
from openai import OpenAI


# ---------------------------------------------------------
# 1. Reload the final extraction prompt and schema
# ---------------------------------------------------------

import extraction_config

importlib.reload(extraction_config)

CommentExtraction = extraction_config.CommentExtraction
EXTRACTION_PROMPT_V1 = extraction_config.EXTRACTION_PROMPT_V1


# Stop before spending money if the latest taxonomy is not loaded.
if "Aesthetics and design" not in str(extraction_config.AspectName):
    raise RuntimeError(
        "Your current extraction_config.py is missing "
        "'Aesthetics and design'. Replace it with the corrected "
        "file, restart the kernel, and run this cell again."
    )


# ---------------------------------------------------------
# 2. Schema for one request containing up to 10 comments
# ---------------------------------------------------------

class CommentBatchExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    extractions: list[CommentExtraction] = Field(
        min_length=1,
        max_length=10,
        description=(
            "Exactly one independent extraction for every input comment."
        ),
    )


# ---------------------------------------------------------
# 3. Settings and file names
# ---------------------------------------------------------

BASE_DIR = Path.cwd()

OLD_RESULTS_FILE = (
    BASE_DIR
    / "reviews_development_120_extractions.jsonl"
)

NEW_RESULTS_FILE = (
    BASE_DIR
    / "reviews_development_120_medium_batch10_extractions.jsonl"
)

NEW_ERRORS_FILE = (
    BASE_DIR
    / "reviews_development_120_medium_batch10_errors.jsonl"
)

USAGE_FILE = (
    BASE_DIR
    / "reviews_development_120_medium_batch10_usage.jsonl"
)

COMPARISON_FILE = (
    BASE_DIR
    / "reviews_development_120_medium_batch10_comparison.csv"
)


MODEL = "gpt-5-mini-2025-08-07"
REASONING_EFFORT = "medium"

COMMENTS_PER_REQUEST = 10
MAX_WORKERS = 10
MAX_RETRIES = 3


# GPT-5 mini standard prices per 1 million tokens.
INPUT_PRICE = 0.25
CACHED_INPUT_PRICE = 0.025
OUTPUT_PRICE = 2.00


if not OLD_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {OLD_RESULTS_FILE.name} "
        f"in {BASE_DIR}."
    )


# ---------------------------------------------------------
# 4. Create the API client
# ---------------------------------------------------------

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )


client_medium_batch = OpenAI(
    max_retries=0,
    timeout=600.0,
)


# ---------------------------------------------------------
# 5. Load the exact same 120 comments
# ---------------------------------------------------------

old_records = []


with OLD_RESULTS_FILE.open(
    "r",
    encoding="utf-8",
) as file:

    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            old_records.append(
                json.loads(line)
            )

        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number} of "
                f"{OLD_RESULTS_FILE.name}: {error}"
            ) from error


source_rows = []


for record in old_records:
    source_rows.append(
        {
            "development_order": int(
                record["development_order"]
            ),

            "comment_id": str(
                record["comment_id"]
            ),

            "year": record.get("year"),

            "listing_activity": record.get(
                "listing_activity"
            ),

            "comments_clean": str(
                record["comments_clean"]
            ),

            "old_extraction": record["extraction"],
        }
    )


df_same_120 = (
    pd.DataFrame(source_rows)
    .sort_values("development_order")
    .reset_index(drop=True)
)


if len(df_same_120) != 120:
    raise ValueError(
        f"Expected 120 comments, "
        f"but found {len(df_same_120)}."
    )


if df_same_120["comment_id"].duplicated().any():
    raise ValueError(
        "Duplicate comment IDs were found."
    )


# ---------------------------------------------------------
# 6. Helper functions
# ---------------------------------------------------------

def nested_value(
    obj,
    *names,
    default=0,
):
    """
    Safely retrieve a nested token-usage value.
    """

    current = obj

    for name in names:
        if current is None:
            return default

        current = getattr(
            current,
            name,
            None,
        )

    if current is None:
        return default

    return current


def calculate_cost(usage):
    """
    Calculate estimated API cost for one request.
    """

    uncached_input = max(
        usage["input_tokens"]
        - usage["cached_input_tokens"],
        0,
    )

    return (
        uncached_input
        * INPUT_PRICE
        / 1_000_000

        + usage["cached_input_tokens"]
        * CACHED_INPUT_PRICE
        / 1_000_000

        + usage["output_tokens"]
        * OUTPUT_PRICE
        / 1_000_000
    )


def make_batches(
    rows,
    batch_size=10,
):
    """
    Divide comment rows into groups of 10.
    """

    batches = []

    for start in range(
        0,
        len(rows),
        batch_size,
    ):
        group = rows[
            start : start + batch_size
        ]

        orders = [
            int(row["development_order"])
            for row in group
        ]

        batches.append(
            {
                "batch_id": (
                    f"orders_"
                    f"{min(orders):03d}_"
                    f"{max(orders):03d}"
                ),

                "rows": group,
            }
        )

    return batches


# ---------------------------------------------------------
# 7. Additional prompt rules for multi-comment requests
# ---------------------------------------------------------

BATCH_INSTRUCTION = """

MULTI-COMMENT REQUEST RULES

The user message contains a JSON object with a "comments" list.
Process each comment independently using all rules above.

Return exactly one CommentExtraction for every input comment.
Copy every comment_id exactly.
Do not omit, duplicate, merge, or combine comments.
Do not use information from one comment to interpret another.
Return the extractions in the same order as the input comments.
"""


SYSTEM_PROMPT_BATCH = (
    EXTRACTION_PROMPT_V1
    + BATCH_INSTRUCTION
)


# ---------------------------------------------------------
# 8. Function for extracting one group of 10 comments
# ---------------------------------------------------------

def extract_one_batch(batch):
    """
    Send one API request containing up to 10 comments.
    """

    rows = batch["rows"]
    batch_id = batch["batch_id"]


    expected_ids = [
        str(row["comment_id"])
        for row in rows
    ]


    payload = {
        "comments": [
            {
                "comment_id": str(
                    row["comment_id"]
                ),

                "comment": str(
                    row["comments_clean"]
                ),
            }

            for row in rows
        ]
    }


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        started = time.perf_counter()

        try:
            completion = (
                client_medium_batch
                .chat
                .completions
                .parse(
                    model=MODEL,

                    reasoning_effort=(
                        REASONING_EFFORT
                    ),

                    messages=[
                        {
                            "role": "system",
                            "content": (
                                SYSTEM_PROMPT_BATCH
                            ),
                        },

                        {
                            "role": "user",
                            "content": json.dumps(
                                payload,
                                ensure_ascii=False,
                            ),
                        },
                    ],

                    response_format=(
                        CommentBatchExtraction
                    ),
                )
            )


            elapsed_seconds = (
                time.perf_counter()
                - started
            )


            message = (
                completion
                .choices[0]
                .message
            )


            if message.parsed is None:
                raise ValueError(
                    "No parsed batch extraction was returned. "
                    f"Refusal: {message.refusal}"
                )


            parsed_batch = message.parsed


            returned_ids = [
                str(item.comment_id)
                for item
                in parsed_batch.extractions
            ]


            if len(returned_ids) != len(expected_ids):
                raise ValueError(
                    f"Expected {len(expected_ids)} "
                    f"extractions, but received "
                    f"{len(returned_ids)}."
                )


            if len(set(returned_ids)) != len(returned_ids):
                raise ValueError(
                    "The response contains duplicate comment IDs."
                )


            if set(returned_ids) != set(expected_ids):

                missing = sorted(
                    set(expected_ids)
                    - set(returned_ids)
                )

                unexpected = sorted(
                    set(returned_ids)
                    - set(expected_ids)
                )

                raise ValueError(
                    f"Comment ID mismatch. "
                    f"Missing={missing}; "
                    f"unexpected={unexpected}."
                )


            extraction_by_id = {
                str(item.comment_id): item

                for item
                in parsed_batch.extractions
            }


            usage_object = completion.usage


            usage = {
                "input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens",
                ),

                "cached_input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens_details",
                    "cached_tokens",
                ),

                "output_tokens": nested_value(
                    usage_object,
                    "completion_tokens",
                ),

                "reasoning_tokens": nested_value(
                    usage_object,
                    "completion_tokens_details",
                    "reasoning_tokens",
                ),

                "total_tokens": nested_value(
                    usage_object,
                    "total_tokens",
                ),
            }


            usage["estimated_cost_usd"] = (
                calculate_cost(usage)
            )


            comment_records = []


            for row in rows:

                comment_id = str(
                    row["comment_id"]
                )

                extraction = extraction_by_id[
                    comment_id
                ]


                comment_records.append(
                    {
                        "development_order": int(
                            row["development_order"]
                        ),

                        "comment_id": comment_id,

                        "year": row.get("year"),

                        "listing_activity": row.get(
                            "listing_activity"
                        ),

                        "comments_clean": str(
                            row["comments_clean"]
                        ),

                        "model": MODEL,

                        "reasoning_effort": (
                            REASONING_EFFORT
                        ),

                        "comments_per_request": (
                            len(rows)
                        ),

                        "batch_id": batch_id,

                        "attempt": attempt,

                        "extraction": (
                            extraction.model_dump()
                        ),
                    }
                )


            return {
                "status": "success",

                "batch_id": batch_id,

                "comment_records": (
                    comment_records
                ),

                "usage_record": {
                    "batch_id": batch_id,

                    "development_orders": [
                        int(row["development_order"])
                        for row in rows
                    ],

                    "comment_ids": expected_ids,

                    "comment_count": len(rows),

                    "elapsed_seconds": (
                        elapsed_seconds
                    ),

                    "attempt": attempt,

                    "model": MODEL,

                    "reasoning_effort": (
                        REASONING_EFFORT
                    ),

                    "usage": usage,
                },
            }


        except openai.AuthenticationError:
            raise


        except Exception as error:

            if attempt < MAX_RETRIES:
                time.sleep(
                    2 ** attempt
                )

                continue


            return {
                "status": "error",

                "batch_id": batch_id,

                "development_orders": [
                    int(row["development_order"])
                    for row in rows
                ],

                "comment_ids": expected_ids,

                "error_type": (
                    type(error).__name__
                ),

                "message": str(error),

                "attempts": attempt,
            }


# ---------------------------------------------------------
# 9. Resume safely if an earlier run stopped
# ---------------------------------------------------------

completed_ids = set()


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if line.strip():
                saved_record = json.loads(
                    line
                )

                completed_ids.add(
                    str(
                        saved_record["comment_id"]
                    )
                )


remaining_rows = df_same_120[
    ~df_same_120["comment_id"].isin(
        completed_ids
    )
].to_dict(
    "records"
)


batches = make_batches(
    remaining_rows,
    COMMENTS_PER_REQUEST,
)


print(
    "Latest extraction_config.py reloaded."
)

print(
    "Prompt characters:",
    len(EXTRACTION_PROMPT_V1),
)

print(
    "Aesthetics and design present: yes"
)

print(
    "Same development comments loaded:",
    len(df_same_120),
)

print(
    "Model:",
    MODEL,
)

print(
    "Reasoning effort:",
    REASONING_EFFORT,
)

print(
    "Comments per request:",
    COMMENTS_PER_REQUEST,
)

print(
    "Parallel requests:",
    MAX_WORKERS,
)

print(
    "Already completed:",
    len(completed_ids),
)

print(
    "Comments remaining:",
    len(remaining_rows),
)

print(
    "API requests to send:",
    len(batches),
)


confirmation = input(
    "Type RUN MEDIUM BATCH10 120 "
    "to start the paid test: "
).strip()


# ---------------------------------------------------------
# 10. Run the 12 API requests in parallel
# ---------------------------------------------------------

run_seconds = None


if confirmation != "RUN MEDIUM BATCH10 120":

    print(
        "API test cancelled."
    )


elif not batches:

    print(
        "All 120 medium-batch results "
        "are already complete."
    )


else:
    run_started = time.perf_counter()

    successful_batches = 0
    error_batches = 0
    successful_comments = 0
    run_cost = 0.0


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                extract_one_batch,
                batch,
            ): batch["batch_id"]

            for batch in batches
        }


        with NEW_RESULTS_FILE.open(
            "a",
            encoding="utf-8",
        ) as results_file, NEW_ERRORS_FILE.open(
            "a",
            encoding="utf-8",
        ) as errors_file, USAGE_FILE.open(
            "a",
            encoding="utf-8",
        ) as usage_file:


            for finished_number, future in enumerate(
                as_completed(futures),
                start=1,
            ):
                result = future.result()


                if result["status"] == "success":

                    for record in result[
                        "comment_records"
                    ]:
                        results_file.write(
                            json.dumps(
                                record,
                                ensure_ascii=False,
                            )
                            + "\n"
                        )

                    results_file.flush()


                    usage_file.write(
                        json.dumps(
                            result["usage_record"],
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    usage_file.flush()


                    successful_batches += 1

                    successful_comments += len(
                        result["comment_records"]
                    )

                    run_cost += (
                        result["usage_record"]
                        ["usage"]
                        ["estimated_cost_usd"]
                    )


                else:
                    error_record = dict(
                        result
                    )

                    error_record.pop(
                        "status"
                    )

                    errors_file.write(
                        json.dumps(
                            error_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    errors_file.flush()

                    error_batches += 1


                print(
                    f"Finished request "
                    f"{finished_number}/"
                    f"{len(batches)} | "
                    f"comments="
                    f"{successful_comments} | "
                    f"batch_errors="
                    f"{error_batches} | "
                    f"cost=${run_cost:.4f}"
                )


    run_seconds = (
        time.perf_counter()
        - run_started
    )


    print(
        "\n--------------- RUN SUMMARY ---------------"
    )

    print(
        "Successful batch requests:",
        successful_batches,
    )

    print(
        "Failed batch requests:",
        error_batches,
    )

    print(
        "Successful comments:",
        successful_comments,
    )

    print(
        "Wall-clock minutes:",
        round(
            run_seconds / 60,
            2,
        ),
    )

    print(
        "Cost for this run: $",
        round(
            run_cost,
            4,
        ),
    )


    if successful_comments:

        throughput = (
            successful_comments
            / run_seconds
        )

        print(
            "Projected time for 5,000 comments:",
            round(
                5_000
                / throughput
                / 3600,
                2,
            ),
            "hours",
        )

        print(
            "Projected time for 50,000 comments:",
            round(
                50_000
                / throughput
                / 3600,
                2,
            ),
            "hours",
        )


# ---------------------------------------------------------
# 11. Load all saved medium-batch results
# ---------------------------------------------------------

new_records = []


if NEW_RESULTS_FILE.exists():

    with NEW_RESULTS_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if line.strip():
                new_records.append(
                    json.loads(line)
                )


# Keep the latest saved result for each comment ID.
new_by_id = {
    str(record["comment_id"]): record

    for record in new_records
}


new_records = sorted(
    new_by_id.values(),

    key=lambda record: (
        record["development_order"]
    ),
)


expected_ids = set(
    df_same_120["comment_id"]
)

returned_ids = set(
    new_by_id
)

missing_ids = (
    expected_ids
    - returned_ids
)


# ---------------------------------------------------------
# 12. Load and deduplicate request-level usage
# ---------------------------------------------------------

usage_records = []


if USAGE_FILE.exists():

    with USAGE_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if line.strip():
                usage_records.append(
                    json.loads(line)
                )


usage_by_batch = {
    record["batch_id"]: record

    for record in usage_records
}


usage_records = list(
    usage_by_batch.values()
)


all_usage = [
    record["usage"]
    for record in usage_records
]


total_cost = sum(
    usage.get(
        "estimated_cost_usd",
        0,
    )
    for usage in all_usage
)


total_input_tokens = sum(
    usage.get(
        "input_tokens",
        0,
    )
    for usage in all_usage
)


total_cached_tokens = sum(
    usage.get(
        "cached_input_tokens",
        0,
    )
    for usage in all_usage
)


total_output_tokens = sum(
    usage.get(
        "output_tokens",
        0,
    )
    for usage in all_usage
)


total_reasoning_tokens = sum(
    usage.get(
        "reasoning_tokens",
        0,
    )
    for usage in all_usage
)


# ---------------------------------------------------------
# 13. Automatic structure and evidence checks
# ---------------------------------------------------------

evidence_checks = []


for record in new_records:

    validated = (
        CommentExtraction.model_validate(
            record["extraction"]
        )
    )


    for finding in validated.findings:

        evidence_checks.append(
            finding.evidence_quote
            in record["comments_clean"]
        )


if evidence_checks:

    evidence_pass_rate = (
        sum(evidence_checks)
        / len(evidence_checks)
    )

else:

    evidence_pass_rate = 1.0


print(
    "\n--------------- "
    "TOTAL MEDIUM-BATCH RESULTS "
    "---------------"
)

print(
    "Returned comments:",
    len(new_records),
    "/ 120",
)

print(
    "Missing comments:",
    len(missing_ids),
)

print(
    "Successful API requests saved:",
    len(usage_records),
)

print(
    "Input tokens:",
    total_input_tokens,
)

print(
    "Cached input tokens:",
    total_cached_tokens,
)

print(
    "Output tokens:",
    total_output_tokens,
)

print(
    "Reasoning tokens:",
    total_reasoning_tokens,
)

print(
    "Measured total cost: $",
    round(
        total_cost,
        4,
    ),
)

print(
    "Exact evidence-quote pass rate:",
    f"{evidence_pass_rate:.1%}",
)


if new_records:

    projected_5k_cost = (
        total_cost
        / len(new_records)
        * 5_000
    )

    projected_50k_cost = (
        total_cost
        / len(new_records)
        * 50_000
    )

    print(
        "Projected cost for 5,000 comments: $",
        round(
            projected_5k_cost,
            2,
        ),
    )

    print(
        "Projected cost for 50,000 comments: $",
        round(
            projected_50k_cost,
            2,
        ),
    )


if missing_ids:

    print(
        "Missing comment IDs:",
        sorted(missing_ids),
    )


# ---------------------------------------------------------
# 14. Create old-versus-medium-batch comparison
# ---------------------------------------------------------

old_by_id = {
    str(row["comment_id"]): row

    for row in source_rows
}


comparison_rows = []


for comment_id in df_same_120[
    "comment_id"
]:

    old_record = old_by_id[
        comment_id
    ]

    new_record = new_by_id.get(
        comment_id
    )


    old_findings = (
        old_record
        ["old_extraction"]
        ["findings"]
    )


    if new_record:

        medium_findings = (
            new_record
            ["extraction"]
            ["findings"]
        )

    else:

        medium_findings = []


    comparison_rows.append(
        {
            "development_order": (
                old_record[
                    "development_order"
                ]
            ),

            "comment_id": comment_id,

            "comments_clean": (
                old_record[
                    "comments_clean"
                ]
            ),

            "old_finding_count": (
                len(old_findings)
            ),

            "medium_batch_finding_count": (
                len(medium_findings)
            ),

            "old_aspects": " | ".join(
                finding["aspect"]

                for finding
                in old_findings
            ),

            "medium_batch_aspects": " | ".join(
                finding["aspect"]

                for finding
                in medium_findings
            ),

            "old_extraction_json": json.dumps(
                old_record[
                    "old_extraction"
                ],
                ensure_ascii=False,
            ),

            "medium_batch_extraction_json": (
                json.dumps(
                    new_record[
                        "extraction"
                    ],
                    ensure_ascii=False,
                )

                if new_record

                else ""
            ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df.to_csv(
    COMPARISON_FILE,
    index=False,
)


print(
    "\nComparison file saved:",
    COMPARISON_FILE.name,
)


# ---------------------------------------------------------
# 15. Display orders connected to reviewed issues
# ---------------------------------------------------------

focus_orders = [
    4,
    5,
    8,
    30,
    31,
    47,
    54,
    55,
    61,
    64,
    77,
    79,
    84,
    85,
    86,
    92,
    96,
    97,
    104,
    111,
    115,
    118,
]


focus_comparison = comparison_df[
    comparison_df[
        "development_order"
    ].isin(
        focus_orders
    )
][
    [
        "development_order",
        "comments_clean",
        "old_aspects",
        "medium_batch_aspects",
        "old_finding_count",
        "medium_batch_finding_count",
    ]
]


focus_comparison

Latest extraction_config.py reloaded.
Prompt characters: 19459
Aesthetics and design present: yes
Same development comments loaded: 120
Model: gpt-5-mini-2025-08-07
Reasoning effort: medium
Comments per request: 10
Parallel requests: 10
Already completed: 0
Comments remaining: 120
API requests to send: 12
Finished request 1/12 | comments=10 | batch_errors=0 | cost=$0.0093
Finished request 2/12 | comments=20 | batch_errors=0 | cost=$0.0203
Finished request 3/12 | comments=30 | batch_errors=0 | cost=$0.0323
Finished request 4/12 | comments=40 | batch_errors=0 | cost=$0.0444
Finished request 5/12 | comments=50 | batch_errors=0 | cost=$0.0593
Finished request 6/12 | comments=60 | batch_errors=0 | cost=$0.0736
Finished request 7/12 | comments=70 | batch_errors=0 | cost=$0.0881
Finished request 8/12 | comments=80 | batch_errors=0 | cost=$0.1026
Finished request 9/12 | comments=90 | batch_errors=0 | cost=$0.1188
Finished request 10/12 | comments=100 | batch_errors=0 | cost=$0.1353
Finished re

,development_order,comments_clean,old_aspects,medium_batch_aspects,old_finding_count,medium_batch_finding_count
3,4,thanks for the stay really enjoyed it will boo...,Other | Overall stay | Overall stay,Overall stay,3,1
4,5,"such a comfortable place to stay, they thought...",Comfort | Amenities | Property condition | Loc...,Comfort | Amenities | Aesthetics and design | ...,4,4
7,8,This place was spacious and clean. Location wa...,Cleanliness | Comfort | Location | Amenities |...,Space and capacity | Cleanliness | Location | ...,14,10
29,30,Amazing space and great frontdesk,Overall stay | Other,Space and capacity | Communication,2,2
30,31,"This place was so beautiful,nice and quiet. Th...",Quietness | Location | Cleanliness | Comfort |...,Aesthetics and design | Quietness | Cleanlines...,8,7
46,47,Our family loved our quick stay here. The hous...,Overall stay | Property condition | Amenities ...,Overall stay | Aesthetics and design | Communi...,6,5
53,54,I adored this place! This historic Inn is abso...,Overall stay | Communication | Amenities | Loc...,Overall stay | Aesthetics and design | Communi...,5,6
54,55,If you are looking for a place to stay in Pasa...,Overall stay | Location | Location | Comfort |...,Overall stay | Aesthetics and design | Comfort...,11,10
60,61,"Wir waren, im Rahmen einer USA Westküsten Rund...",Overall stay | Check-in | Amenities | Accuracy...,Overall stay | Check-in | Amenities | Accuracy...,9,9
63,64,Host was very responsive. Place was nice and q...,Communication | Quietness | Overall stay | Other,Communication | Quietness,4,2


In [ ]:
#Test results
#120/120 comments returned
#12 API requests
#10 comments per request
#0 failed batches
#Total cost: $0.1579
#Projected 50,000-comment cost: $65.78
#427 findings
#Exact evidence-quote rate: 422/427 = 98.8%
#So the goal cannot be:Every extraction must be unquestionably perfect.
#The real goal is:The method is consistent, reproducible, accurate enough for 
#aggregate business analysis, and its remaining error rate is measured.
#At this point, continuing to revise the prompt using the same 120 comments 
#creates a new risk: overfitting. We may fix one phrase while making another 
#boundary worse.